In [0]:
# Limpar cache temporário do Spark
spark.catalog.clearCache()
print("Cache do Spark limpo.")

In [0]:
# Remover dados de teste anteriores
# Preserva todas as pastas e subpastas
# Remove somente arquivos

BASE_PATH = (
    "/Volumes/test/default/"
    "dados_origin_visao_unica/resultados"
)

PASTAS = [
    "prices",
    "prices_historic",
    "prices_penultimate_execution"
]

def limpar_arquivos(caminho):
    total = 0

    for item in dbutils.fs.ls(caminho):

        if item.isDir():
            # Entra na subpasta sem excluí-la
            total += limpar_arquivos(item.path)

        else:
            # Remove somente o arquivo
            dbutils.fs.rm(item.path)
            print(f"[REMOVIDO] {item.path}")
            total += 1

    return total


total_geral = 0

for pasta in PASTAS:

    caminho = f"{BASE_PATH}/{pasta}/"

    print(f"\n[PROCESSANDO] {pasta}")

    try:
        quantidade = limpar_arquivos(caminho)
        total_geral += quantidade

        print(f"[CONCLUÍDO] {quantidade} arquivos removidos")

    except Exception as erro:
        print(f"[ERRO] {pasta}: {erro}")


print("\n" + "=" * 55)
print("LIMPEZA FINALIZADA")
print(f"TOTAL DE ARQUIVOS REMOVIDOS: {total_geral}")
print("ESTRUTURA DE DIRETÓRIOS PRESERVADA")
print("=" * 55)

In [0]:
# Setar volumes do databricks
ORIGEM = '/Volumes/test/default/dados_origin_visao_unica/parquet'
VOLUME = "/Volumes/test/default/dados_origin_visao_unica/parquet_volumetria_final"

print("ORIGEM:", ORIGEM)
print("VOLUME:", VOLUME)

In [0]:

# CARREGAR BIBLIOTECAS SNESS, INICIALIZAR E IDENTIFICAR VERSOES
# ============================================================

import os
import sys
import types
import importlib
import importlib.util
import tomllib

# ============================================================
# 1. IDENTIFICAR USUARIO E DIRETORIOS
# ============================================================

current_user = spark.sql(
    "SELECT current_user()"
).first()[0]

BASE = (
    f"/Workspace/Users/{current_user}/"
    "Magalu-visao-unica-precos"
)

COMMON_ROOT = (
    f"{BASE}/03_CONFIGURAÇÕES/"
    "BIBLIOTECA_SNESS/sness-common-main"
)

SPARK_ROOT = (
    f"{BASE}/03_CONFIGURAÇÕES/"
    "BIBLIOTECA_SNESS/sness-spark-main"
)

COMMON = f"{COMMON_ROOT}/sness"
SPARK = f"{SPARK_ROOT}/sness"

# ============================================================
# 2. VALIDAR DIRETORIOS
# ============================================================

for nome, caminho in [
    ("SNESS COMMON", COMMON),
    ("SNESS SPARK", SPARK)
]:
    if not os.path.isdir(caminho):
        raise FileNotFoundError(
            f"{nome} nao encontrado: {caminho}"
        )

# ============================================================
# 3. CONFIGURAR AMBIENTE
# ============================================================

# Habilitar somente se as adaptacoes da SNESS para
# Serverless estiverem sendo utilizadas neste Compute.
# Esta flag nao altera o tipo real do Compute.
os.environ["SNESS_DATABRICKS_SERVERLESS"] = "true"

# ============================================================
# 4. CONFIGURAR NAMESPACE SNESS
# ============================================================

sness = types.ModuleType("sness")
sness.__path__ = [COMMON, SPARK]

sys.modules["sness"] = sness

# ============================================================
# 5. CONFIGURAR DATALAKE
# ============================================================

datalake = types.ModuleType("sness.datalake")

datalake.__path__ = [
    f"{COMMON}/datalake",
    f"{SPARK}/datalake"
]

sys.modules["sness.datalake"] = datalake
sness.datalake = datalake

# ============================================================
# 6. CONFIGURAR ETL
# ============================================================

etl = types.ModuleType("sness.etl")

etl.__path__ = [
    p
    for p in [
        f"{COMMON}/etl",
        f"{SPARK}/etl"
    ]
    if os.path.isdir(p)
]

sys.modules["sness.etl"] = etl
sness.etl = etl

# ============================================================
# 7. CONFIGURAR SETTINGS
# ============================================================

settings_init = f"{SPARK}/settings/__init__.py"

if not os.path.isfile(settings_init):
    raise FileNotFoundError(
        f"Settings SNESS nao encontrado: {settings_init}"
    )

spec = importlib.util.spec_from_file_location(
    "sness.settings",
    settings_init,
    submodule_search_locations=[
        f"{COMMON}/settings",
        f"{SPARK}/settings"
    ]
)

settings = importlib.util.module_from_spec(spec)

sys.modules["sness.settings"] = settings
sness.settings = settings

spec.loader.exec_module(settings)

# ============================================================
# 8. IMPORTAR COMPONENTES SNESS
# ============================================================

from sness.datalake.metadata import (
    Zone,
    Environment
)

from sness.datalake.sness_spark import SnessSpark

# ============================================================
# 9. INICIALIZAR SNESS SPARK
# ============================================================

print("Inicializando SNESS Spark...")

ss = SnessSpark()

if not hasattr(ss, "read"):
    raise RuntimeError(
        "SNESS inicializada sem o atributo read."
    )

if not hasattr(ss.read, "parquet"):
    raise RuntimeError(
        "SNESS inicializada sem o metodo read.parquet."
    )

print("SNESS Spark inicializada com sucesso.")

# ============================================================
# 10. IDENTIFICAR VERSOES
# ============================================================

def obter_versao_sness(diretorio):

    arquivo = os.path.join(
        diretorio,
        "pyproject.toml"
    )

    if not os.path.isfile(arquivo):
        return "Nao identificada"

    with open(arquivo, "rb") as f:
        config = tomllib.load(f)

    return (
        config.get("tool", {})
              .get("poetry", {})
              .get("version")
        or config.get("project", {})
                 .get("version")
        or "Nao identificada"
    )

versao_common = obter_versao_sness(COMMON_ROOT)
versao_spark = obter_versao_sness(SPARK_ROOT)

# ============================================================
# 11. VALIDAR INICIALIZACAO
# ============================================================

validacoes = {
    "Spark": spark is not None,
    "SNESS Common": os.path.isdir(COMMON),
    "SNESS Spark": os.path.isdir(SPARK),
    "Environment": Environment is not None,
    "Zone": Zone is not None,
    "SNESS ss": ss is not None,
    "Leitor Parquet": hasattr(ss.read, "parquet")
}

# ============================================================
# 12. RESULTADO FINAL
# ============================================================

print()
print("=" * 60)
print("BIBLIOTECAS SNESS - INICIALIZACAO")
print("=" * 60)

print(f"Sness Common : {versao_common}")
print(f"Sness Spark  : {versao_spark}")
print(f"Apache Spark : {spark.version}")
print(f"Usuario      : {current_user}")
print(f"Classe ss    : {type(ss).__name__}")

print("-" * 60)

for componente, resultado in validacoes.items():
    status = "OK" if resultado else "ERRO"
    print(f"[{status}] {componente}")

print("=" * 60)

if all(validacoes.values()):
    print("SNESS INICIALIZADA E LEITOR DISPONIVEL")
else:
    raise RuntimeError(
        "Falha na validacao dos componentes SNESS."
    )


In [0]:
# CONFIGURACAO DO MODULO UTILS
# ============================================================

import os
import sys
import importlib

current_user = spark.sql(
    "SELECT current_user()"
).collect()[0][0]

BASE = f"/Workspace/Users/{current_user}/Magalu-visao-unica-precos"

RAIZ_PROJETO = f"{BASE}/01_CODIGO_AS_IS/visaounicapreco"

DIRETORIO_UTILS = (
    f"{RAIZ_PROJETO}/visao_unica_commons/etls/utils"
)

ARQUIVO_UTILS = f"{DIRETORIO_UTILS}/utils.py"

print("===== CONFIGURACAO UTILS =====")

if not os.path.isfile(ARQUIVO_UTILS):
    raise FileNotFoundError(ARQUIVO_UTILS)

# Configurar caminhos
for caminho in [DIRETORIO_UTILS, RAIZ_PROJETO]:
    if caminho in sys.path:
        sys.path.remove(caminho)
    sys.path.insert(0, caminho)

# Atualizar cache de importacao
importlib.invalidate_caches()

# Importar modulo original
import utils

print("Status: OK")
print("Modulo:", utils.__name__)
print("Arquivo:", utils.__file__)

In [0]:
# Validação Utils
import sys

print("===== VALIDACAO UTILS =====")

assert "utils" in sys.modules, "Modulo utils nao carregado"

import utils

print("Status: OK")
print("Arquivo:", utils.__file__)

In [0]:
# Shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "auto")

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)


In [0]:
# Monitoramento spark
import logging

# Reduz logs Python
for nome in ("py4j", "pyspark", "grpc"):
    logging.getLogger(nome).setLevel(logging.ERROR)

# Reduz logs Spark em clusters tradicionais
try:
    spark.sparkContext.setLogLevel("ERROR")
except (AttributeError, RuntimeError, NotImplementedError):
    pass

# Desativa métricas opcionais de streaming
try:
    spark.conf.set("spark.sql.streaming.metricsEnabled", "false")
except Exception:
    pass

print("[OK] Redução de logs e métricas opcionais aplicada")

In [0]:
# EXECUÇÃO DO SCRIPT - VISAO UNICA DE PRECOS - AS IS
# DATABRICKS - 4 CORES CPUs - 20 WORKERS
# ============================================================

import os
import sys
import time
import importlib.util
import traceback
import io
import re
from contextlib import redirect_stdout
from datetime import datetime
from datetime import timezone

# ============================================================
# 1. CONFIGURACOES
# ============================================================

os.environ["SNESS_DATABRICKS_SERVERLESS"] = "true"

AMBIENTE = "prd"

current_user = spark.sql(
    "SELECT current_user()"
).collect()[0][0]

BASE = (
    f"/Workspace/Users/{current_user}/"
    "Magalu-visao-unica-precos"
)

RAIZ_PROJETO = (
    f"{BASE}/01_CODIGO_AS_IS/visaounicapreco"
)

ARQUIVO = (
    f"{RAIZ_PROJETO}/visao_unica_preco/"
    "etls/trusted/visao_precos.py"
)

VOLUME_ORIGEM = (
    "/Volumes/test/default/dados_origin_visao_unica/"
    "parquet_volumetria_final"
)

VOLUME_RESULTADOS = (
    "/Volumes/test/default/dados_origin_visao_unica/"
    "resultados"
)

LARGURA = 82

# ============================================================
# 2. FORMATACAO DA SAIDA
# ============================================================

def linha(caractere="="):
    print(caractere * LARGURA, flush=True)


def titulo(texto):
    print()
    linha()
    print(f" {texto}", flush=True)
    linha()


def formatar_tempo(segundos):
    horas = int(segundos // 3600)
    minutos = int((segundos % 3600) // 60)
    restante = segundos % 60

    return f"{horas:02d}:{minutos:02d}:{restante:06.3f}"


def formatar_log(mensagem, destino):
    """
    Formata as mensagens sem utilizar print().
    Escreve diretamente na saida original do notebook.
    """

    mensagem = mensagem.strip()

    if not mensagem:
        return

    # ----------------------------------------
    # LEITURA SNESS
    # ----------------------------------------

    if "[SNESS READ REAL]" in mensagem:

        dataset = re.search(
            r"dataset=([^\s|]+)",
            mensagem
        )

        caminho = re.search(
            r"path=(.+)",
            mensagem
        )

        nome = (
            dataset.group(1)
            if dataset
            else "DESCONHECIDO"
        )

        path = (
            caminho.group(1)
            if caminho
            else ""
        )

        destino.write(
            f"  [LEITURA]  {nome}\n"
            f"             {path}\n"
        )

        destino.flush()
        return

    # ----------------------------------------
    # GRAVACAO SNESS
    # ----------------------------------------

    if "[SNESS WRITE REAL]" in mensagem:

        dataset = re.search(
            r"dataset=([^\s|]+)",
            mensagem
        )

        caminho = re.search(
            r"path=(.+)",
            mensagem
        )

        nome = (
            dataset.group(1)
            if dataset
            else "DESCONHECIDO"
        )

        path = (
            caminho.group(1)
            if caminho
            else ""
        )

        destino.write(
            f"  [GRAVACAO] {nome}\n"
            f"             {path}\n"
        )

        destino.flush()
        return

    # ----------------------------------------
    # OUTRAS MENSAGENS
    # ----------------------------------------

    destino.write(
        f"  [INFO]     {mensagem}\n"
    )

    destino.flush()


# ============================================================
# 3. INTERCEPTADOR DE LOGS
# ============================================================

class SaidaFormatada(io.TextIOBase):

    def __init__(self, destino):
        super().__init__()
        self.destino = destino
        self.buffer = ""

    def write(self, texto):

        if not isinstance(texto, str):
            texto = str(texto)

        self.buffer += texto

        while "\n" in self.buffer:

            linha_log, self.buffer = (
                self.buffer.split("\n", 1)
            )

            if linha_log.strip():
                formatar_log(
                    linha_log,
                    self.destino
                )

        return len(texto)

    def flush(self):

        if self.buffer.strip():

            formatar_log(
                self.buffer,
                self.destino
            )

        self.buffer = ""
        self.destino.flush()


# ============================================================
# 4. VALIDAR CAMINHOS
# ============================================================

if not os.path.isfile(ARQUIVO):

    raise FileNotFoundError(
        f"Script nao encontrado: {ARQUIVO}"
    )

if RAIZ_PROJETO not in sys.path:

    sys.path.insert(
        0,
        RAIZ_PROJETO
    )

DIRETORIO_UTILS = (
    f"{RAIZ_PROJETO}/"
    "visao_unica_preco/commons/utils"
)

if os.path.isfile(
    f"{DIRETORIO_UTILS}/utils.py"
):

    if DIRETORIO_UTILS not in sys.path:

        sys.path.insert(
            0,
            DIRETORIO_UTILS
        )

# ============================================================
# 5. INICIALIZACAO
# ============================================================

inicio_utc = datetime.now(timezone.utc)
inicio = time.perf_counter()

titulo(
    "VISAO UNICA DE PRECOS | EXECUCAO AS IS"
)

print(f" Ambiente       : {AMBIENTE}")
print(
    f" Inicio         : "
    f"{datetime.now():%d/%m/%Y %H:%M:%S}"
)
print(f" Script         : {ARQUIVO}")
print(f" Volume origem  : {VOLUME_ORIGEM}")
print(f" Resultados     : {VOLUME_RESULTADOS}")

# ============================================================
# 6. CARREGAR SCRIPT ORIGINAL
# ============================================================

spec = importlib.util.spec_from_file_location(
    "visao_precos_original",
    ARQUIVO
)

if spec is None or spec.loader is None:

    raise ImportError(
        f"Nao foi possivel carregar: {ARQUIVO}"
    )

modulo = importlib.util.module_from_spec(
    spec
)

sys.modules[spec.name] = modulo

spec.loader.exec_module(modulo)

# ============================================================
# 7. EXECUTAR PIPELINE
# ============================================================

titulo("PROCESSAMENTO DOS DATASETS")

status = "EM EXECUCAO"

# Captura o stdout original do Databricks.
saida_original = sys.stdout

# O interceptador escreve diretamente neste destino,
# evitando recursao infinita.
saida_formatada = SaidaFormatada(
    saida_original
)

try:

    with redirect_stdout(saida_formatada):

        modulo.main([
            "--environment",
            AMBIENTE
        ])

    status = "SUCESSO"

except Exception:

    status = "ERRO"

    print()
    print("[ERRO] Falha na execucao do pipeline.")
    traceback.print_exc()

    raise

finally:

    saida_formatada.flush()

    tempo_total = (
        time.perf_counter() - inicio
    )

    # Disponibilizar as metricas para a proxima celula
    METRICAS_ULTIMA_EXECUCAO = {
        "run_id": f"databricks__{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        "inicio_utc": datetime.fromtimestamp(
            time.time() - tempo_total,
            tz=__import__("datetime").timezone.utc
        ),
        "fim_utc": datetime.now(__import__("datetime").timezone.utc),
        "run_total_min": round(tempo_total / 60, 3),
        "status": status
    }

    titulo("RESUMO FINAL DA EXECUCAO")

    print(f" Status         : {status}")

    print(
        f" Fim            : "
        f"{datetime.now():%d/%m/%Y %H:%M:%S}"
    )

    print(
        f" TEMPO TOTAL    : "
        f"{formatar_tempo(tempo_total)}"
    )

    linha()


In [0]:
# INVENTARIO DE DATASETS - VISAO UNICA DE PRECOS
# ============================================================

import os
from datetime import datetime
from pyspark.sql import Row

# ============================================================
# 1. CONFIGURACOES
# ============================================================

BASE = "/Volumes/test/default/dados_origin_visao_unica"

DIRETORIOS = {
    "DADOS SINTÉTICOS": f"{BASE}/parquet_volumetria_final",
    "RESULTADOS": f"{BASE}/resultados"
}

# ============================================================
# 2. FUNCOES AUXILIARES
# ============================================================

def formatar_numero(valor):
    return f"{valor:,}".replace(",", ".")


def formatar_tamanho(bytes_total):

    if bytes_total >= 1024 ** 3:
        return f"{bytes_total / (1024 ** 3):,.1f} GB".replace(",", "X").replace(".", ",").replace("X", ".")

    if bytes_total >= 1024 ** 2:
        return f"{bytes_total / (1024 ** 2):,.1f} MB".replace(",", "X").replace(".", ",").replace("X", ".")

    return f"{bytes_total / 1024:,.1f} KB".replace(",", "X").replace(".", ",").replace("X", ".")


def listar_arquivos(caminho):

    arquivos = []

    for raiz, diretorios, nomes in os.walk(caminho):

        for nome in nomes:

            if nome.lower().endswith(".parquet"):

                arquivo = os.path.join(raiz, nome)

                arquivos.append(arquivo)

    return arquivos


def identificar_particoes(caminho):

    particoes = set()

    for raiz, diretorios, arquivos in os.walk(caminho):

        relativo = os.path.relpath(raiz, caminho)

        if relativo == ".":
            continue

        partes = relativo.split(os.sep)

        for parte in partes:

            if "=" in parte:
                particoes.add(parte)

    if not particoes:
        return "nenhuma"

    return f"{len(particoes)} partições"


# ============================================================
# 3. INVENTARIO POR DIRETORIO
# ============================================================

for tipo, diretorio in DIRETORIOS.items():

    print()
    print("=" * 100)
    print(f" DATASETS - {tipo}")
    print("=" * 100)

    if not os.path.isdir(diretorio):

        print(f"[AVISO] Diretório não encontrado: {diretorio}")
        continue

    registros = []

    datasets = sorted(os.listdir(diretorio))

    for dataset in datasets:

        caminho = os.path.join(diretorio, dataset)

        if not os.path.isdir(caminho):
            continue

        try:

            # Quantidade real de registros
            linhas = spark.read.parquet(caminho).count()

            # Arquivos Parquet
            arquivos = listar_arquivos(caminho)

            quantidade_arquivos = len(arquivos)

            # Tamanho físico
            tamanho_bytes = sum(
                os.path.getsize(arquivo)
                for arquivo in arquivos
            )

            # Partições
            particoes = identificar_particoes(caminho)

            registros.append({
                "Dataset": dataset,
                "Linhas": formatar_numero(linhas),
                "Arquivos": formatar_numero(quantidade_arquivos),
                "Tamanho": formatar_tamanho(tamanho_bytes),
                "Partições": particoes,
                "Mudança última revisão": "Não avaliado",
                "_ordem": linhas
            })

            print(f"[OK] {dataset}", flush=True)

        except Exception as erro:

            print(f"[ERRO] {dataset}: {str(erro)[:180]}")

    # ========================================================
    # 4. EXIBIR TABELA
    # ========================================================

    if registros:

        registros = sorted(
            registros,
            key=lambda x: x["_ordem"],
            reverse=True
        )

        for item in registros:
            del item["_ordem"]

        display(
            spark.createDataFrame(registros)
            .select(
                "Dataset",
                "Linhas",
                "Arquivos",
                "Tamanho",
                "Partições",
                "Mudança última revisão"
            )
        )

    else:
        print("Nenhum dataset encontrado.")


In [0]:
# METRICAS DA EXECUCAO - VISAO UNICA DE PRECOS
# ============================================================

import pandas as pd

if "METRICAS_ULTIMA_EXECUCAO" not in globals():
    raise RuntimeError(
        "Nenhuma execucao instrumentada encontrada. "
        "Execute primeiro a celula do pipeline."
    )

metricas = METRICAS_ULTIMA_EXECUCAO

def formatar_data(valor):
    return valor.strftime("%Y-%m-%d %H:%M:%S")

def formatar_minutos(valor):
    if valor is None:
        return "N/D"
    return f"{valor:.3f}".replace(".", ",")

# ============================================================
# TABELA DA ULTIMA EXECUCAO
# ============================================================

resultado = pd.DataFrame([{
    "run_id": metricas["run_id"],

    "inicio_trusted_prices_utc": formatar_data(
        metricas["inicio_utc"]
    ),

    "fim_publisher_utc": "N/D",

    "trusted_prices_min": formatar_minutos(
        metricas["run_total_min"]
    ),

    "bq_prices_min": "N/D",

    "publisher_min": "N/D",

    "run_total_min": formatar_minutos(
        metricas["run_total_min"]
    ),

    "disparo_ate_fim_min": "N/D",

    "maquina_executor_trusted": "N/D",

    "pico_executores_trusted": "N/D",

    "custo_estimado_trusted_brl": "N/D"
}])

display(resultado)


In [0]:
# INVENTARIO DE DEPENDENCIAS
# REQUIREMENTS | PYPROJECT | AMBIENTE DE EXECUCAO
# ============================================================

import os
import sys
import platform
import socket
import importlib.util
import importlib.metadata as md
from pathlib import Path

try:
    import tomllib
except ImportError:
    import tomli as tomllib

from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

# ============================================================
# 1. CONFIGURACOES
# ============================================================

current_user = spark.sql(
    "SELECT current_user()"
).collect()[0][0]

PROJETO = Path(
    f"/Workspace/Users/{current_user}/"
    "Magalu-visao-unica-precos"
)

REQUIREMENTS_PATH = (
    PROJETO
    / "01_CODIGO_AS_IS"
    / "visaounicapreco"
    / "visao_unica_preco"
    / "requirements.txt"
)

SNESS_BASE = (
    PROJETO
    / "03_CONFIGURAÇÕES"
    / "BIBLIOTECA_SNESS"
)

PYPROJECT_SPARK = (
    SNESS_BASE / "sness-spark-main" / "pyproject.toml"
)

PYPROJECT_COMMON = (
    SNESS_BASE / "sness-common-main" / "pyproject.toml"
)

PACOTES_ADICIONAIS = [
    "delta-spark",
    "pyarrow",
    "setuptools",
    "wheel",
    "boto3",
    "google-auth",
    "prometheus-client",
    "grpcio",
    "grpcio-status",
    "protobuf",
    "google-cloud-storage",
    "google-cloud-secret-manager",
]

PACOTES_PRIORITARIOS = [
    "google-cloud-storage",
    "google-cloud-secret-manager",
    "numpy",
    "pandas",
    "pyspark",
    "sness-lib",
    "sness-common",
    "sness-spark",
    "apache-spark (runtime)",
    "python",
]

MODULOS = [
    "sness",
    "sness.datalake",
    "sness.datalake.metadata",
    "sness.datalake.sness_spark",
    "sness.etl.spark_utils",
    "utils",
]

# ============================================================
# 2. FUNCOES AUXILIARES
# ============================================================

def normalizar(nome):
    return canonicalize_name(nome)


def versao_instalada(nome):
    try:
        return md.version(nome)
    except md.PackageNotFoundError:
        return "NAO INSTALADO"


def formatar(valor):
    if valor is None or valor == "":
        return "-"
    return str(valor)


def ler_requirements(caminho):

    resultado = {}

    if not caminho.is_file():
        print(f"[AVISO] Arquivo nao encontrado: {caminho}")
        return resultado

    for numero, linha in enumerate(
        caminho.read_text(encoding="utf-8-sig").splitlines(), 1
    ):

        linha = linha.strip()

        if not linha or linha.startswith("#"):
            continue

        linha = linha.split(" #", 1)[0].strip()

        try:
            req = Requirement(linha)
            resultado[normalizar(req.name)] = (
                str(req.specifier) or "-"
            )
        except Exception:
            print(
                f"[AVISO] Requirements linha {numero}: {linha}"
            )

    return resultado


def ler_pyproject(caminho):

    if not caminho.is_file():
        print(f"[AVISO] Arquivo nao encontrado: {caminho}")
        return None, {}

    with caminho.open("rb") as arquivo:
        config = tomllib.load(arquivo)

    projeto = config.get("project", {})

    versao = projeto.get("version")
    dependencias = {}

    for item in projeto.get("dependencies", []):

        try:
            req = Requirement(item)
            dependencias[normalizar(req.name)] = (
                str(req.specifier) or "-"
            )
        except Exception:
            print(f"[AVISO] Dependencia nao interpretada: {item}")

    if projeto.get("requires-python"):
        dependencias["python"] = projeto["requires-python"]

    return versao, dependencias


def verificar_modulo(nome):

    if nome in sys.modules:
        return "DISPONIVEL"

    try:
        return (
            "DISPONIVEL"
            if importlib.util.find_spec(nome) is not None
            else "NAO DISPONIVEL"
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return "NAO DISPONIVEL"


def spark_runtime():

    try:
        return spark.version
    except Exception:
        return "NAO IDENTIFICADO"


def obter_config_spark(chave):

    try:
        return spark.conf.get(chave, None)
    except Exception:
        return None


def obter_memoria_gb():

    try:
        import psutil
        return round(
            psutil.virtual_memory().total / (1024 ** 3),
            2
        )
    except Exception:
        return None


def exibir_tabela(registros, colunas):

    if not registros:
        print("Nenhum registro encontrado.")
        return

    # Pandas evita iniciar um job Spark apenas
    # para apresentar os dados do inventario.
    import pandas as pd

    display(
        pd.DataFrame(registros, columns=colunas)
    )


# ============================================================
# 3. LER DEPENDENCIAS DO PROJETO
# ============================================================

requirements = ler_requirements(REQUIREMENTS_PATH)

versao_spark, deps_spark = ler_pyproject(
    PYPROJECT_SPARK
)

versao_common, deps_common = ler_pyproject(
    PYPROJECT_COMMON
)

pyproject = {}

for pacote in set(deps_spark) | set(deps_common):

    valor_spark = deps_spark.get(pacote)
    valor_common = deps_common.get(pacote)

    if valor_spark and valor_common:

        if valor_spark == valor_common:
            pyproject[pacote] = valor_spark
        else:
            pyproject[pacote] = (
                f"Spark: {valor_spark} | "
                f"Common: {valor_common}"
            )

    else:
        pyproject[pacote] = valor_spark or valor_common

if versao_spark:
    pyproject["sness-spark"] = versao_spark

if versao_common:
    pyproject["sness-common"] = versao_common


# ============================================================
# 4. IDENTIFICAR VERSOES DO AMBIENTE
# ============================================================

def versao_ambiente(pacote):

    if pacote == "sness-common":

        versao = versao_instalada("sness-lib-common")

        if versao != "NAO INSTALADO":
            return versao

        return (
            f"{versao_common} (codigo-fonte)"
            if versao_common
            else "NAO INSTALADO"
        )

    if pacote == "sness-spark":

        versao = versao_instalada("sness-lib-spark")

        if versao != "NAO INSTALADO":
            return versao

        return (
            f"{versao_spark} (codigo-fonte)"
            if versao_spark
            else "NAO INSTALADO"
        )

    return versao_instalada(pacote)


# ============================================================
# 5. INVENTARIO DE DEPENDENCIAS
# ============================================================

pacotes = set(requirements)
pacotes.update(pyproject)
pacotes.update(map(normalizar, PACOTES_ADICIONAIS))

pacotes.discard("python")

prioritarios = {
    normalizar(p)
    for p in PACOTES_PRIORITARIOS
    if p not in ("apache-spark (runtime)", "python")
}

pacotes_comuns = sorted(
    p for p in pacotes
    if p not in prioritarios
)

registros_dependencias = []


def adicionar_pacote(pacote):

    registros_dependencias.append({
        "Pacote / Modulo": pacote,
        "Requirements": formatar(requirements.get(pacote)),
        "Pyproject": formatar(pyproject.get(pacote)),
        "Databricks": versao_ambiente(pacote)
    })


for pacote in pacotes_comuns:
    adicionar_pacote(pacote)

for pacote in PACOTES_PRIORITARIOS:

    if pacote == "apache-spark (runtime)":

        registros_dependencias.append({
            "Pacote / Modulo": pacote,
            "Requirements": formatar(
                requirements.get("pyspark")
            ),
            "Pyproject": "-",
            "Databricks": spark_runtime()
        })

    elif pacote == "python":

        registros_dependencias.append({
            "Pacote / Modulo": pacote,
            "Requirements": "-",
            "Pyproject": formatar(pyproject.get("python")),
            "Databricks": platform.python_version()
        })

    else:
        adicionar_pacote(normalizar(pacote))


# ============================================================
# 6. DISPONIBILIDADE DOS MODULOS
# ============================================================

registros_modulos = []

for modulo in MODULOS:

    registros_modulos.append({
        "Modulo": modulo,
        "Versao": "-",
        "Disponibilidade": verificar_modulo(modulo)
    })


# ============================================================
# 7. INFORMACOES DA MAQUINA / COMPUTACAO
# ============================================================

cluster_id = os.environ.get(
    "DATABRICKS_CLUSTER_ID"
)

cluster_name = os.environ.get(
    "DATABRICKS_CLUSTER_NAME"
)

runtime_version = os.environ.get(
    "DATABRICKS_RUNTIME_VERSION"
)

cpu_disponivel = os.cpu_count()
memoria_gb = obter_memoria_gb()

# Configuracoes Spark, quando expostas ao notebook
executor_instances = obter_config_spark(
    "spark.executor.instances"
)

executor_cores = obter_config_spark(
    "spark.executor.cores"
)

executor_memory = obter_config_spark(
    "spark.executor.memory"
)

dynamic_max = obter_config_spark(
    "spark.dynamicAllocation.maxExecutors"
)

registros_ambiente = [
    {
        "Indicador": "Cluster ID",
        "Valor": cluster_id or "NAO DISPONIVEL"
    },
    {
        "Indicador": "Cluster Name",
        "Valor": cluster_name or "NAO DISPONIVEL"
    },
    {
        "Indicador": "Databricks Runtime",
        "Valor": runtime_version or "NAO DISPONIVEL"
    },
    {
        "Indicador": "Spark Runtime",
        "Valor": spark_runtime()
    },
    {
        "Indicador": "Python",
        "Valor": platform.python_version()
    },
    {
        "Indicador": "Sistema operacional",
        "Valor": platform.platform()
    },
    {
        "Indicador": "Hostname Python",
        "Valor": socket.gethostname()
    },
    {
        "Indicador": "CPUs visiveis ao processo Python",
        "Valor": formatar(cpu_disponivel)
    },
    {
        "Indicador": "Memoria visivel ao processo Python (GB)",
        "Valor": formatar(memoria_gb)
    },
    {
        "Indicador": "Spark executor instances (config)",
        "Valor": formatar(executor_instances)
    },
    {
        "Indicador": "Spark executor cores (config)",
        "Valor": formatar(executor_cores)
    },
    {
        "Indicador": "Spark executor memory (config)",
        "Valor": formatar(executor_memory)
    },
    {
        "Indicador": "Spark dynamic max executors (config)",
        "Valor": formatar(dynamic_max)
    },
]

# ============================================================
# 8. EXIBIR INVENTARIOS
# ============================================================

print()
print("=" * 100)
print(" INVENTARIO DE DEPENDENCIAS - VISAO UNICA DE PRECOS")
print("=" * 100)

print(f" Requirements : {REQUIREMENTS_PATH}")
print(f" SNESS Spark  : {PYPROJECT_SPARK}")
print(f" SNESS Common : {PYPROJECT_COMMON}")

print()
print(" DEPENDENCIAS - REQUIREMENTS X PYPROJECT X DATABRICKS")
print("-" * 100)

exibir_tabela(
    registros_dependencias,
    [
        "Pacote / Modulo",
        "Requirements",
        "Pyproject",
        "Databricks"
    ]
)

print()
print("=" * 100)
print(" DISPONIBILIDADE DOS MODULOS - SNESS - PYTHON")
print("=" * 100)

exibir_tabela(
    registros_modulos,
    [
        "Modulo",
        "Versao",
        "Disponibilidade"
    ]
)

print()
print("=" * 100)
print(" AMBIENTE DE EXECUCAO - DATABRICKS")
print("=" * 100)

exibir_tabela(
    registros_ambiente,
    [
        "Indicador",
        "Valor"
    ]
)
